In [355]:
import pytesseract

pytesseract.pytesseract.tesseract_cmd = r"/opt/homebrew/bin/tesseract"

In [356]:
import os
import re
import pdfplumber
import pandas as pd
import pytesseract
from pdf2image import convert_from_path

pytesseract.pytesseract.tesseract_cmd = r"/opt/homebrew/bin/tesseract"
POPPLER_PATH = "/opt/homebrew/bin"

In [357]:
def extract_all_text(file_path):
    file_path = os.path.abspath(file_path)

    text = ""

    import tempfile
    import subprocess

    try:

        portfolio_check = subprocess.run(
            ["/opt/homebrew/bin/pdfdetach", "-list", file_path],
            capture_output=True,
            text=True
        )

        if "embedded files" in portfolio_check.stdout.lower() and "0 embedded files" not in portfolio_check.stdout.lower():

            with tempfile.TemporaryDirectory() as tmpdir:

                subprocess.run(
                    ["/opt/homebrew/bin/pdfdetach", "-saveall", file_path],
                    cwd=tmpdir,
                    check=False
                )

                for embedded_file in os.listdir(tmpdir):

                    if embedded_file.lower().endswith(".pdf"):

                        embedded_path = os.path.join(tmpdir, embedded_file)

                        with pdfplumber.open(embedded_path) as embedded_pdf:
                            for page in embedded_pdf.pages:
                                page_text = page.extract_text()

                                if page_text:
                                    text += "\n" + page_text

                        images = convert_from_path(
                            embedded_path,
                            poppler_path="/opt/homebrew/bin"
                        )

                        for image in images:
                            ocr_text = pytesseract.image_to_string(image)
                            text += "\n" + ocr_text

            return text

    except Exception as e:
        print("Portfolio extraction failed:", e)

    try:

        with pdfplumber.open(file_path) as pdf:

            for page in pdf.pages:

                page_text = page.extract_text()

                if page_text:
                    text += "\n" + page_text

    except Exception as e:
        print("pdfplumber failed:", e)

    try:

        images = convert_from_path(
            file_path,
            poppler_path="/opt/homebrew/bin"
        )

        for image in images:

            ocr_text = pytesseract.image_to_string(image)

            text += "\n" + ocr_text

    except Exception as e:
        print("OCR failed:", e)

    return text

In [358]:
def clean_insurer(value):

    if value is None:
        return None

    value = re.sub(r"\s+", " ", value).strip()
    value = re.sub(r"^bond was executed by\s+", "", value, flags=re.IGNORECASE)
    value = re.sub(r"\s+", " ", value).strip()
    value = re.sub(r"\b[A-Z]{3}\s+\d{1,2}\s+\d{4}\b", "", value)
    value = re.sub(r"\s+", " ", value).strip()
    value = value.replace("RL!", "RLI")

    known = [
        "RLI Insurance Company",
        "Liberty Mutual Insurance Company",
        "Pennsylvania Insurance Company",
        "Travelers Casualty and Surety Company of America",
        "Traveler’s Casualty and Surety Company of America",
        "Travelers Casualty and Surety Company"
    ]

    for name in known:
        if name.lower() in value.lower():
            return name

    return value.strip().rstrip(",")

In [400]:
def extract_fields(text):

    amount_patterns = [
        r"in the amount of\s+(\$[\d,]+(?:\.\d{2})?)",
        r"amount of\s+(\$[\d,]+(?:\.\d{2})?)",
        r"Bond amount\s*:?\s*(\$[\d,]+(?:\.\d{2})?)",
        r"Bond Amount\s*:?\s*(\$?[\d,]+(?:\.\d{2})?)" 
    ]

    insurer_patterns = [
        r"executed by\s+([A-Z][A-Za-z\s&.'’,-]*?Surety Company\s+of\s+America),?\s+as\s+the\s+[Ss]urety",
        r"executed by\s+([A-Z][A-Za-z\s&.'’,-]*?Surety Company\s+of\s+America)\s+as\s+[Ss]urety",
        r"executed by\s+([A-Z][A-Za-z\s&.'’,-]*?Insurance Company)\s+as\s+[Ss]urety",
        r"([A-Z][A-Za-z\s&.'’,-]*?(?:Insurance|Surety)\s+Company(?:\s+of\s+America)?),\s*as\s+[Ss]urety",
        r"executed by the surety,\s*([A-Z][A-Za-z\s&.'’,-]*?Company(?: of America)?),\s*on",
        r"The Surety is the Company Guaranteeing Performance\.\s*([A-Z][A-Za-z\s&.'’,-]*?Insurance Company(?: of America)?)\s+Name of Surety",
        r"Name of Surety\s*[:;]\s*([A-Z][A-Za-z\s&.'’,-]*?(?:Insurance|Surety)\s+Company(?:\s+of\s+America)?)",
        r"Surety:\s*([^\n]+)"
    ]

    company_patterns = [
        r"acknowledges receipt of\s+(?:the\s+)?([A-Z][A-Za-z0-9\s&.'’,-]{2,120}?(?:LLC|Inc\.?|Corporation|Company))['’]s",
        r"Name of Principal\s*[:;]\s*_?\s*([A-Z][A-Za-z0-9\s&.'’,-]{2,120}?(?:LLC|Inc\.?|Corporation|Company))",
        r"by the\s+principal,\s*([A-Z][A-Za-z0-9\s&.'’,-]{2,120}?(?:LLC|Inc\.?|Corporation|Company)),\s*on",
        r"principal,\s*([A-Z][A-Za-z0-9\s&.'’,-]{2,120}?(?:LLC|Inc\.?|Corporation|Company)),\s*on",
        r"([A-Z][A-Za-z0-9&.'’,-]{1,40}(?:\s+[A-Z][A-Za-z0-9&.'’,-]{1,40}){0,8}\s+(?:LLC|Inc\.?|Corporation|Company)),?\s+as\s+Principal",
        r"Principal:\s*([^\n]+)"
        
    ]

    date_patterns = [
        r"namely\s*([A-Za-z]+\s+\d{1,2},\s*\d{4})",
        r"effective as of the date filed,\s*namely\s*([A-Za-z]+\s+\d{1,2},\s*\d{4})",
        r"Effective date:\s*([A-Za-z]+\s+\d{1,2},\s+\d{4})",
        r"effective as of the date filed,\s*([A-Za-z]+\s+\d{1,2},\s+\d{4})",
        r"considered effective\s*([A-Za-z]+\s+\d{1,2},\s+\d{4})",
        r"effective\s*_*\s*([A-Za-z]+)\s*(\d{1,2})[.,]?\s*(\d{4})"       
    ]

    def clean_value(value):
        if value is None:
            return None
        value = re.sub(r"\s+", " ", value)
        value = value.strip("_ ").strip()
        value = re.sub(r"\b[A-Z]{3}\s+\d{1,2}\s+\d{4}\b", "", value)
        value = re.sub(r"\s+", " ", value).strip()
        value = re.sub(r"^(bond\s+)?was executed by\s+", "", value, flags=re.IGNORECASE)
        value = value.rstrip(".,;:")
        value = re.sub(r"^[Tt]he\s+", "", value)
        value = re.sub(r"\bInc\b$", "Inc.", value)
        if value.lower().startswith("and "):
            value = value[4:]
        if value == "$100.00":
            value = "$100,000.00"
        if value == "$1":
            value = "$100,000.00"
        value = value.replace("Enerov", "Energy")
        value = value.replace("GSOE 1", "GSOE I")
        value = value.replace("Atiantic", "Atlantic")
        value = re.sub(r"\bvangrid", "Avangrid", value)
        value = re.sub(r"A+vangrid", "Avangrid", value)
        value = re.sub(r"\binvenergy", "Invenergy", value)
        return value

    def is_bad_value(value):
        if value is None:
            return True

        value = clean_value(value)

        bad_words = [
            "Mailing Address",
            "PO Box",
            "P.O. Box",
            "Address",
            "OMB Control",
            "Expiration Date",
            "Regional BOEM",
            "Bond Type",
            "Check here",
            "Schedule A",
            "conditioned to cover",
            "lease OCS"
        ]

        for bad in bad_words:
            if bad.lower() in value.lower():
                return True

        if len(value) < 3:
            return True

        if len(value) > 160:
            return True

        if "____" in value:
            return True

        if value.count("_") > 3:
            return True

        return False

    def find_first(patterns, text, is_date=False):
        for pattern in patterns:
            match = re.search(pattern, text, re.IGNORECASE | re.DOTALL)
            if match:
                if is_date and len(match.groups()) == 3:
                    value = f"{match.group(1)} {match.group(2)}, {match.group(3)}"
                else:
                    value = match.group(1)

                value = clean_value(value)

                if not is_bad_value(value):
                    return value

        return None

    result = {
        "Company Name": find_first(company_patterns, text),
        "Bond Amount": find_first(amount_patterns, text),
        "Insurance Company Name": clean_insurer(find_first(insurer_patterns, text)),
        "Effective Date of the Bond": find_first(date_patterns, text, is_date=True),
        "Project Code": None
    }

    project_match = re.search(r"OCS-[AP]\s*\d{4}", text)

    if project_match:
        result["Project Code"] = project_match.group(0)

    if result["Bond Amount"] is not None and not result["Bond Amount"].startswith("$"):
        result["Bond Amount"] = "$" + result["Bond Amount"]

    if "letter of credit" in text.lower() or re.search(r"\bLOC\b", text):

        if result["Company Name"] is None:
            m = re.search(r"on behalf of\s+([A-Z][A-Za-z0-9\s,&.\-]*?LLC)", text)
            if m and not is_bad_value(m.group(1)):
                result["Company Name"] = clean_value(m.group(1))

        if result["Company Name"] is None:
            m = re.search(r"The Lessee,\s*([A-Z][A-Za-z0-9\s&.'’,-]*?LLC)", text)
            if m and not is_bad_value(m.group(1)):
                result["Company Name"] = clean_value(m.group(1))

        if result["Company Name"] is None:
            m = re.search(r"The Lessee is changed from\s+[A-Z][A-Za-z0-9\s&.'’,-]*?LLC\s+to\s+([A-Z][A-Za-z0-9\s&.'’,-]*?LLC)", text)
            if m and not is_bad_value(m.group(1)):
                result["Company Name"] = clean_value(m.group(1))

        if result["Insurance Company Name"] is None:
            m = re.search(r"issued by\s+(.+?),\s+to meet", text)
            if m and not is_bad_value(m.group(1)):
                result["Insurance Company Name"] = clean_value(m.group(1))
        if result["Insurance Company Name"] is None:
            m = re.search(
                r"by\s+the\s+surety,\s*([A-Za-z\s&,\n]+?Company(?:\s+of\s+America)?)",
                text,
                re.IGNORECASE
            )
            if m:
                result["Insurance Company Name"] = clean_value(m.group(1))

        if result["Insurance Company Name"] is None:
            m = re.search(r"issued.*?by\s+([A-Z][A-Za-z0-9\s&.'’,-]*?Bank[A-Za-z0-9\s&.'’,-]*?Branch)", text, re.IGNORECASE | re.DOTALL)
            if m and not is_bad_value(m.group(1)):
                result["Insurance Company Name"] = clean_value(m.group(1))

        m = re.search(r"Bond Amount\s*[:;]\s*\$?([\d,]+(?:\.\d{2})?)", text)
        if m:
            result["Bond Amount"] = clean_value(m.group(1))

        if result["Effective Date of the Bond"] is None:
            m = re.search(r"accepted(?:\s+and\s+is)?\s+effective\s+as\s+of\s+([A-Za-z]+\s+\d{1,2},\s+\d{4})", text)
            if m:
                result["Effective Date of the Bond"] = clean_value(m.group(1))

        if result["Effective Date of the Bond"] is None:
            m = re.search(r"effective as of the date filed,\s*([A-Za-z]+\s+\d{1,2},\s+\d{4})", text)
            if m:
                result["Effective Date of the Bond"] = clean_value(m.group(1))
    
    if result["Company Name"] is None:

        m = re.search(
            r"(?:Principal(?:'s)? Name|Settlor)\s*[:\-]?\s*([A-Z][A-Za-z0-9\s&.',-]*?(?:LLC|Inc\.?))",
            text,
            re.IGNORECASE
        )

        if m and not is_bad_value(m.group(1)):
            result["Company Name"] = clean_value(m.group(1))
    
    if "Principal Name is hereby amended" in text:
        m = re.search(r"\bTo:\s*([A-Z][A-Za-z0-9\s&.'’,-]*?LLC)", text)
        if m and not is_bad_value(m.group(1)):
            result["Company Name"] = clean_value(m.group(1))
    
    if result["Company Name"] is None:
        m = re.search(r"Name of Principal\s*[:;]\s*([^\n]+)", text)
        if m and not is_bad_value(m.group(1)):
            result["Company Name"] = clean_value(m.group(1))

    if result["Insurance Company Name"] is None:

        m = re.search(
            r"([A-Z][A-Za-z\s&.',-]*?Insurance Company)",
            text
        )

        if m and not is_bad_value(m.group(1)):
            result["Insurance Company Name"] = clean_insurer(m.group(1))
        
    if result["Insurance Company Name"] is None:
        m = re.search(r"Name of Surety:\s*([^\n]+)", text)
        if m and not is_bad_value(m.group(1)):
            result["Insurance Company Name"] = clean_value(m.group(1))
    if result["Insurance Company Name"] is None:
        m = re.search(r"ty:\s*([A-Z][A-Za-z\s&.'’,-]*?Insurance Company)", text)
        if m and not is_bad_value(m.group(1)):
            result["Insurance Company Name"] = clean_value(m.group(1))

    if result["Bond Amount"] is None or result["Bond Amount"] in ["1", "$1"]:
        m = re.search(r"Bond Amount\s*[:;]\s*\$?([\d ,]+(?:\.\d{2})?)", text)

        if m:
            amount = clean_value(m.group(1))
            amount = amount.replace(" ", "")

            digits = re.sub(r"\D", "", amount)

            if len(digits) >= 5:
                result["Bond Amount"] = "$" + amount
    text_lower = text.lower()

    if (
        result["Effective Date of the Bond"] is None
        and (
            "made effective" in text_lower
            or "replacement bond" in text_lower
            or "effective as of" in text_lower
        )
    ):
        result["Effective Date of the Bond"] = "MANUAL REVIEW"
    if (
        result["Bond Amount"] is None
        and "available amount" in text.lower()
        and "from" in text.lower()
        and "to" in text.lower()
    ):

        m = re.search(
            r"available amount\s+from\s+\$?([\d,]+(?:\.\d{2})?)\s+to\s+\$?([\d,]+(?:\.\d{2})?)",
            text,
            re.IGNORECASE
        )

        if m:
            result["Bond Amount"] = "$" + m.group(2)
    
    if "bond amount" in text.lower() and "to" in text.lower():

        m = re.search(
            r"from\s+\$?[\d,]+(?:\.\d{2})?\s+to\s+\$?([\d,]+(?:\.\d{2})?)",
            text,
            re.IGNORECASE
        )

        if m:
            result["Bond Amount"] = "$" + m.group(1)
    
    if "bond rider" in text.lower() and (
        result["Company Name"] is None
        or "bond rider" in result["Company Name"].lower()
    ):

        m = re.search(
            r"\bon behalf of\s+[_\s]*([A-Za-z][A-Za-z0-9\s&.',-]*?(?:LLC|Inc\.?))",
            text,
            re.IGNORECASE
        )

        if m and not is_bad_value(m.group(1)):
            result["Company Name"] = clean_value(m.group(1))

    if result["Company Name"] is None and "Vineyard Mid-Atlantic LLC" in text:
        result["Company Name"] = "Vineyard Mid-Atlantic LLC"


    if result["Company Name"] is None and "South Fork Wind, LLC" in text:
        result["Company Name"] = "South Fork Wind, LLC"

    if result["Insurance Company Name"] is None and "Nordea Bank Abp" in text:
        result["Insurance Company Name"] = "Nordea Bank Abp, New York Branch"


    if (
        "amendment" in text.lower()
        and "letter of credit" in text.lower()
        and "from" in text.lower()
        and "to" in text.lower()
    ):

        m = re.search(
            r"from\s+\$?([\d,]+(?:\.\d{2})?)\s+to\s+\$?([\d,]+(?:\.\d{2})?)",
            text,
            re.IGNORECASE
        )

        if m:
            result["Bond Amount"] = "$" + m.group(2)
    
    if result["Insurance Company Name"] is None:
        m = re.search(
            r"by\s+the\s+surety,\s*([A-Za-z\s&]+?Company(?:\s+of\s+America)?)\s*,?\s+on\s+[A-Z][a-z]+\s+\d{1,2},\s+\d{4}",
            text,
            re.IGNORECASE | re.DOTALL
        )

        if m:
            result["Insurance Company Name"] = clean_value(m.group(1))


    if result["Company Name"] is None:
        m = re.search(
            r"Mr\.\s+[A-Za-z\s.]+\s+([A-Z][A-Za-z0-9\s&.'’,-]*?(?:Wind LLC|LLC|Inc\.?|Company))\s+c/o",
            text,
            re.IGNORECASE | re.DOTALL
        )

        if m and not is_bad_value(m.group(1)):
            result["Company Name"] = clean_value(m.group(1))
    
    if result["Bond Amount"] is None:
        m = re.search(
            r"Lessee[’']s\s+\$?([\d,]+(?:\.\d{2})?)\s+supplemental financial assurance",
            text,
            re.IGNORECASE
        )

        if m:
            result["Bond Amount"] = "$" + m.group(1)


        if result["Company Name"] in [None, "LLC", "Wind LLC"] and "Sunrise Wind LLC" in text:
            result["Company Name"] = "Sunrise Wind LLC"



    if result["Company Name"] is None:
        m = re.search(
            r"for the account of\s+([A-Z][A-Za-z0-9\s&.'’,-]+?(?:LLC|Inc\.?|Company))",
            text,
            re.IGNORECASE | re.DOTALL
        )

        if m and not is_bad_value(m.group(1)):
            result["Company Name"] = clean_value(m.group(1))


    if result["Insurance Company Name"] is None:
        m = re.search(
            r"LOC\)\s+issued\s+by\s+([A-Z][A-Za-z0-9\s&.,'’,-]+?Branch)",
            text,
            re.IGNORECASE | re.DOTALL
        )

        if m and not is_bad_value(m.group(1)):
            result["Insurance Company Name"] = clean_value(m.group(1))




    if result["Company Name"] is None or "bond rider dated" in result["Company Name"].lower():
        m = re.search(
            r"to be attached to\s+([A-Z][A-Za-z0-9\s&.'’,-]+?(?:LLC|Inc\.?|Company))['’]s",
            text,
            re.IGNORECASE | re.DOTALL
        )

        if m and not is_bad_value(m.group(1)):
            result["Company Name"] = clean_value(m.group(1))

    
    if "principal" in text.lower() and "name has changed from" in text.lower():
        m = re.search(
            r"Principal[’']?s\s+name\s+has\s+changed\s+from:\s*([A-Z][A-Za-z0-9\s&.'’,-]*?(?:LLC|Inc\.?|Company))\s*To:\s*([A-Z][A-Za-z0-9\s&.'’,-]*?(?:LLC|Inc\.?|Company))",
            text,
            re.IGNORECASE | re.DOTALL
        )

        if m and not is_bad_value(m.group(2)):
            result["Company Name"] = clean_value(m.group(2))
    
    if result["Company Name"] is None and "lessee" in text.lower() and "name from" in text.lower():
        m = re.search(
            r"Lessee[’']?s\s+name\s+from\s+[A-Z][A-Za-z0-9\s&.'’,-]*?(?:LLC|Inc\.?|Company)\s+to\s+([A-Z][A-Za-z0-9\s&.'’,-]*?(?:LLC|Inc\.?|Company))",
            text,
            re.IGNORECASE | re.DOTALL
        )

        if m and not is_bad_value(m.group(1)):
            result["Company Name"] = clean_value(m.group(1))


    if result["Company Name"] is None:
        m = re.search(
            r"Principal\s+Name\s+to:\s*([A-Z][A-Za-z0-9\s&.'’,-]+?(?:LLC|Inc\.?|Company))",
            text,
            re.IGNORECASE | re.DOTALL
        )

        if m and not is_bad_value(m.group(1)):
            result["Company Name"] = clean_value(m.group(1))


    if result["Company Name"] is None and "Virginia Electric and Power Company" in text:
        result["Company Name"] = "Virginia Electric and Power Company"


    if (
        result["Company Name"] is not None
        and "as surety and" in result["Company Name"]
        and "SouthCoast Wind Energy LLC" in text
    ):
        result["Company Name"] = "SouthCoast Wind Energy LLC"

    if (
        result["Company Name"] is not None
        and "OW Ocean Winds East" in result["Company Name"]
        and "Bluepoint Wind, LLC" in text
    ):
        result["Company Name"] = "Bluepoint Wind, LLC"


    if result["Bond Amount"] is None:
        m = re.search(
            r"in the amount\s+of\s+(\$[\d,]+(?:\.\d{2})?)",
            text,
            re.IGNORECASE
        )

        if m:
            result["Bond Amount"] = clean_value(m.group(1))
    
    if result["Company Name"] is None and "Virginia Electric and Power Company" in text:
        result["Company Name"] = "Virginia Electric and Power Company"
    
    for key in result:
        if isinstance(result[key], str):
            result[key] = clean_value(result[key])

    
    if result["Effective Date of the Bond"] in [None, "None", "MANUAL REVIEW"]:
        m = re.search(
            r"effective\s+the\s+\d{1,2}(?:st|nd|rd|th)?\s+day\s+of\s+([A-Za-z]+)\s*,?\s*(\d{4})",
            text,
            re.IGNORECASE | re.DOTALL
        )

        if m:
            month = m.group(1)
            year = m.group(2)

            day_match = re.search(
                r"effective\s+the\s+(\d{1,2})(?:st|nd|rd|th)?\s+day\s+of",
                text,
                re.IGNORECASE
            )

            if day_match:
                day = day_match.group(1)
                result["Effective Date of the Bond"] = f"{month} {day}, {year}"
    
    
    if result["Effective Date of the Bond"] in [None, "None", "MANUAL REVIEW"]:
        m = re.search(
            r"ended\s+on\s+([A-Z][a-z]+\s+\d{1,2},\s+\d{4})",
            text,
            re.IGNORECASE | re.DOTALL
        )

        if m:
            result["Effective Date of the Bond"] = m.group(1)
    
    if result["Company Name"] is None:
        m = re.search(
            r"name\s+of\s+the\s+Lessee\s+from\s+[A-Za-z0-9\s&.,'-]+?\s+to\s+([A-Za-z0-9\s&.,'-]+?(?:LLC|Inc\.?))",
            text,
            re.IGNORECASE | re.DOTALL
        )

        if m and not is_bad_value(m.group(1)):
            result["Company Name"] = clean_value(m.group(1))
    
    return result
    
    
    


In [401]:
pdf_files = [f for f in os.listdir() if f.lower().endswith(".pdf")]

results = []

for file in pdf_files:

    text = extract_all_text(file)

    fields = extract_fields(text)

    fields["File Name"] = file
    fields["Text Length"] = len(text)

    results.append(fields)

df = pd.DataFrame(results)

display(df)



,Company Name,Bond Amount,Insurance Company Name,Effective Date of the Bond,Project Code,File Name,Text Length
0,"Atlantic Shores Offshore Wind, LLC","$50,000.00",Westchester Fire Insurance Company,None,OCS-A 0499,603.pdf,1357
1,"Atlantic Shores Offshore Wind, LLC","$340,529.50",Federal Insurance Company,None,OCS-A 0499,602.pdf,1387
2,US Wind Inc.,"$100,000",Berkley Insurance Company,"February 27, 2018",OCS-A 0490,714 (1).pdf,1819
3,"Deepwater Wind New England, LLC","$201,756.00",RLI Insurance Company,"July 11, 2018",OCS-A 0487,710.pdf,31807
4,"DWW Rev I, LLC","$347,394.00","Nordea Bank Abp, New York Branch","February 3, 2020",OCS-A 0486,706.pdf,4582
5,US Wind Inc.,"$100,000",Berkley Insurance Company,"February 27, 2018",OCS-A 0490,714.pdf,1819
6,Virginia Electric and Power Company,"$434,397",None,MANUAL REVIEW,OCS-A 0483,701.pdf,4600
7,SouthCoast Wind Energy LLC,"$100,000.00",Argonaut Insurance Company,"January 26, 2023",OCS-A 0521,681.pdf,2503
8,Sunrise Wind LLC,"$433,000.00","Nordea Bank Abp, New York Branch","August 7, 2020",OCS-A 0487,709.pdf,4736
9,"Bluepoint Wind, LLC","$100,000",Argonaut Insurance Company,"September 22, 2022",OCS-A 0537,641.pdf,6915
